# 健康監測 RO：功能驗收與固定設定復測
逐格執行，不要 Run All。新硬體 ID 為 ROH2；每批暖機後先檢查 1024 點，再保存 1024 點。沿用 25 MHz、divider 512。先確認你的獨占時段與舊 kernel 已關閉。這是研究用健康監測候選版本，不是已認證 TRNG。

In [1]:
print("kernel OK", flush=True)
import json, platform, pynq, zipfile
from pathlib import Path
from entropy_health_pynq import EntropySession, HealthFailure, validate_artifacts, save_capture
from entropy_campaign import collect_campaign, export_datasets
print("PYNQ", pynq.__version__, "Python", platform.python_version())

kernel OK


PYNQ 2.5 Python 3.6.5


## 1. 離線檢查與操作資料
確認實際板號及操作日期。校驗失敗不能跳過。這格不操作硬體。

In [2]:
board_id = "21600"
experiment_date = "2026-11-11"
print(validate_artifacts("entropy_health.bit"))
print("Board", board_id, "Date", experiment_date)

{'clock_mhz': 25, 'base': 1136656384, 'capacity': 1024, 'channels': 8, 'build_id': 'entropy-health-20261110T172050Z'}
Board 21600 Date 2026-11-11


## 2. 載入健康監測電路一次
應看到四個檢查點，最後 RO OFF。不要在這一步載入舊 entropy_ro.bit。

In [3]:
session = EntropySession("entropy_health.bit")

1/4 health release and clock/reset/AXI wiring checked
2/4 HWH parsed; downloading health overlay (RO OFF)
3/4 25 MHz verified; checking bounded register interface
4/4 health ID/config verified; RO OFF; ready


## 3. 第一段：16 批功能驗收
成功時每批 status=194（0xc2）、startup_seen=1024、observed=2048、count=1024，且三個故障遮罩與 RO enable 都為零。警報即停止並保存診斷；不自動清故障重跑。這段沒有在實板強制注入故障，故障分支以本機模擬驗證。

In [4]:
from datetime import datetime, timezone
import uuid
functional_folder = Path("health_functional") / (experiment_date + "_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8])
functional_folder.mkdir(parents=True, exist_ok=False)
functional_ok = False
try:
    for i in range(16):
        capture = session.capture(divider=512)
        save_capture(capture, functional_folder / ("capture_{:03d}".format(i+1)), board_id)
        print("HEALTH {}/16".format(i+1), capture["health"], flush=True)
    functional_ok = True
except HealthFailure as exc:
    (functional_folder / "fault.json").write_text(json.dumps(exc.snapshot, indent=2), encoding="utf-8")
    raise
finally:
    try:
        session.close()
        print("RO OFF confirmed", flush=True)
    except BaseException as exc:
        (functional_folder / "cleanup_error.json").write_text(json.dumps({"error":repr(exc)}), encoding="utf-8")
        raise
assert functional_ok
print("Functional folder:", str(functional_folder))

HEALTH 1/16 {'status': 194, 'count': 1024, 'ro_enable': 0, 'fault_mask': 0, 'rct_mask': 0, 'apt_mask': 0, 'observed': 2048, 'startup_seen': 1024}
HEALTH 2/16 {'status': 194, 'count': 1024, 'ro_enable': 0, 'fault_mask': 0, 'rct_mask': 0, 'apt_mask': 0, 'observed': 2048, 'startup_seen': 1024}
HEALTH 3/16 {'status': 194, 'count': 1024, 'ro_enable': 0, 'fault_mask': 0, 'rct_mask': 0, 'apt_mask': 0, 'observed': 2048, 'startup_seen': 1024}
HEALTH 4/16 {'status': 194, 'count': 1024, 'ro_enable': 0, 'fault_mask': 0, 'rct_mask': 0, 'apt_mask': 0, 'observed': 2048, 'startup_seen': 1024}
HEALTH 5/16 {'status': 194, 'count': 1024, 'ro_enable': 0, 'fault_mask': 0, 'rct_mask': 0, 'apt_mask': 0, 'observed': 2048, 'startup_seen': 1024}
HEALTH 6/16 {'status': 194, 'count': 1024, 'ro_enable': 0, 'fault_mask': 0, 'rct_mask': 0, 'apt_mask': 0, 'observed': 2048, 'startup_seen': 1024}
HEALTH 7/16 {'status': 194, 'count': 1024, 'ro_enable': 0, 'fault_mask': 0, 'rct_mask': 0, 'apt_mask': 0, 'observed': 2048, 

## 4. 第二段：1000 批固定設定資料復測
只有第一段成功、RO 關閉且仍在你的使用時段，才明確執行本格。此版本增加了啟動健康取樣，也重新實作，不能沿用舊版 6.65 的估計。每批的 1024 個啟動樣本不輸出；原始 payload 全部保存。本格不重新下載電路。

In [5]:
assert functional_ok, "先完成第一段"
campaign = collect_campaign(session, board_id, experiment_date, output_root="health_assessment", target_batches=1000)
print(campaign)
assert campaign["status"] == "complete" and campaign["batches"] == 1000
assert campaign["ro_off_confirmed"]

Campaign folder: health_assessment/2026-11-11_20261110T192028Z_7e1391ad
Saving to health_assessment/2026-11-11_20261110T192028Z_7e1391ad/segments/board_21600/2026-11-11_20261110T192028Z_7ec61aa5
CAPTURE 1/16 round=1 divider=512 samples=1024 RO=OFF
CAPTURE 2/16 round=2 divider=512 samples=1024 RO=OFF
CAPTURE 3/16 round=3 divider=512 samples=1024 RO=OFF
CAPTURE 4/16 round=4 divider=512 samples=1024 RO=OFF
CAPTURE 5/16 round=5 divider=512 samples=1024 RO=OFF
CAPTURE 6/16 round=6 divider=512 samples=1024 RO=OFF
CAPTURE 7/16 round=7 divider=512 samples=1024 RO=OFF
CAPTURE 8/16 round=8 divider=512 samples=1024 RO=OFF
CAPTURE 9/16 round=9 divider=512 samples=1024 RO=OFF
CAPTURE 10/16 round=10 divider=512 samples=1024 RO=OFF
CAPTURE 11/16 round=11 divider=512 samples=1024 RO=OFF
CAPTURE 12/16 round=12 divider=512 samples=1024 RO=OFF
CAPTURE 13/16 round=13 divider=512 samples=1024 RO=OFF
CAPTURE 14/16 round=14 divider=512 samples=1024 RO=OFF
CAPTURE 15/16 round=15 divider=512 samples=1024 RO=OF

CAPTURE 16/16 round=16 divider=512 samples=1024 RO=OFF
CAMPAIGN 128/1000 batches saved; RO OFF=True
Saving to health_assessment/2026-11-11_20261110T192028Z_7e1391ad/segments/board_21600/2026-11-11_20261110T192041Z_7a2f7bb4
CAPTURE 1/16 round=1 divider=512 samples=1024 RO=OFF
CAPTURE 2/16 round=2 divider=512 samples=1024 RO=OFF
CAPTURE 3/16 round=3 divider=512 samples=1024 RO=OFF
CAPTURE 4/16 round=4 divider=512 samples=1024 RO=OFF
CAPTURE 5/16 round=5 divider=512 samples=1024 RO=OFF
CAPTURE 6/16 round=6 divider=512 samples=1024 RO=OFF
CAPTURE 7/16 round=7 divider=512 samples=1024 RO=OFF
CAPTURE 8/16 round=8 divider=512 samples=1024 RO=OFF
CAPTURE 9/16 round=9 divider=512 samples=1024 RO=OFF
CAPTURE 10/16 round=10 divider=512 samples=1024 RO=OFF
CAPTURE 11/16 round=11 divider=512 samples=1024 RO=OFF
CAPTURE 12/16 round=12 divider=512 samples=1024 RO=OFF
CAPTURE 13/16 round=13 divider=512 samples=1024 RO=OFF
CAPTURE 14/16 round=14 divider=512 samples=1024 RO=OFF
CAPTURE 15/16 round=15 di

CAPTURE 16/16 round=16 divider=512 samples=1024 RO=OFF
CAMPAIGN 256/1000 batches saved; RO OFF=True
Saving to health_assessment/2026-11-11_20261110T192028Z_7e1391ad/segments/board_21600/2026-11-11_20261110T192053Z_fa848984
CAPTURE 1/16 round=1 divider=512 samples=1024 RO=OFF
CAPTURE 2/16 round=2 divider=512 samples=1024 RO=OFF
CAPTURE 3/16 round=3 divider=512 samples=1024 RO=OFF
CAPTURE 4/16 round=4 divider=512 samples=1024 RO=OFF
CAPTURE 5/16 round=5 divider=512 samples=1024 RO=OFF
CAPTURE 6/16 round=6 divider=512 samples=1024 RO=OFF
CAPTURE 7/16 round=7 divider=512 samples=1024 RO=OFF
CAPTURE 8/16 round=8 divider=512 samples=1024 RO=OFF
CAPTURE 9/16 round=9 divider=512 samples=1024 RO=OFF
CAPTURE 10/16 round=10 divider=512 samples=1024 RO=OFF
CAPTURE 11/16 round=11 divider=512 samples=1024 RO=OFF
CAPTURE 12/16 round=12 divider=512 samples=1024 RO=OFF
CAPTURE 13/16 round=13 divider=512 samples=1024 RO=OFF
CAPTURE 14/16 round=14 divider=512 samples=1024 RO=OFF
CAPTURE 15/16 round=15 di

CAPTURE 16/16 round=16 divider=512 samples=1024 RO=OFF
CAMPAIGN 384/1000 batches saved; RO OFF=True
Saving to health_assessment/2026-11-11_20261110T192028Z_7e1391ad/segments/board_21600/2026-11-11_20261110T192105Z_06f073e0
CAPTURE 1/16 round=1 divider=512 samples=1024 RO=OFF
CAPTURE 2/16 round=2 divider=512 samples=1024 RO=OFF
CAPTURE 3/16 round=3 divider=512 samples=1024 RO=OFF
CAPTURE 4/16 round=4 divider=512 samples=1024 RO=OFF
CAPTURE 5/16 round=5 divider=512 samples=1024 RO=OFF
CAPTURE 6/16 round=6 divider=512 samples=1024 RO=OFF
CAPTURE 7/16 round=7 divider=512 samples=1024 RO=OFF
CAPTURE 8/16 round=8 divider=512 samples=1024 RO=OFF
CAPTURE 9/16 round=9 divider=512 samples=1024 RO=OFF
CAPTURE 10/16 round=10 divider=512 samples=1024 RO=OFF
CAPTURE 11/16 round=11 divider=512 samples=1024 RO=OFF
CAPTURE 12/16 round=12 divider=512 samples=1024 RO=OFF
CAPTURE 13/16 round=13 divider=512 samples=1024 RO=OFF
CAPTURE 14/16 round=14 divider=512 samples=1024 RO=OFF
CAPTURE 15/16 round=15 di

CAPTURE 16/16 round=16 divider=512 samples=1024 RO=OFF
CAMPAIGN 512/1000 batches saved; RO OFF=True
Saving to health_assessment/2026-11-11_20261110T192028Z_7e1391ad/segments/board_21600/2026-11-11_20261110T192118Z_f7957681
CAPTURE 1/16 round=1 divider=512 samples=1024 RO=OFF
CAPTURE 2/16 round=2 divider=512 samples=1024 RO=OFF
CAPTURE 3/16 round=3 divider=512 samples=1024 RO=OFF
CAPTURE 4/16 round=4 divider=512 samples=1024 RO=OFF
CAPTURE 5/16 round=5 divider=512 samples=1024 RO=OFF
CAPTURE 6/16 round=6 divider=512 samples=1024 RO=OFF
CAPTURE 7/16 round=7 divider=512 samples=1024 RO=OFF
CAPTURE 8/16 round=8 divider=512 samples=1024 RO=OFF
CAPTURE 9/16 round=9 divider=512 samples=1024 RO=OFF
CAPTURE 10/16 round=10 divider=512 samples=1024 RO=OFF
CAPTURE 11/16 round=11 divider=512 samples=1024 RO=OFF
CAPTURE 12/16 round=12 divider=512 samples=1024 RO=OFF
CAPTURE 13/16 round=13 divider=512 samples=1024 RO=OFF
CAPTURE 14/16 round=14 divider=512 samples=1024 RO=OFF
CAPTURE 15/16 round=15 di

CAPTURE 16/16 round=16 divider=512 samples=1024 RO=OFF
CAMPAIGN 640/1000 batches saved; RO OFF=True
Saving to health_assessment/2026-11-11_20261110T192028Z_7e1391ad/segments/board_21600/2026-11-11_20261110T192131Z_80c928b6
CAPTURE 1/16 round=1 divider=512 samples=1024 RO=OFF
CAPTURE 2/16 round=2 divider=512 samples=1024 RO=OFF
CAPTURE 3/16 round=3 divider=512 samples=1024 RO=OFF
CAPTURE 4/16 round=4 divider=512 samples=1024 RO=OFF
CAPTURE 5/16 round=5 divider=512 samples=1024 RO=OFF
CAPTURE 6/16 round=6 divider=512 samples=1024 RO=OFF
CAPTURE 7/16 round=7 divider=512 samples=1024 RO=OFF
CAPTURE 8/16 round=8 divider=512 samples=1024 RO=OFF
CAPTURE 9/16 round=9 divider=512 samples=1024 RO=OFF
CAPTURE 10/16 round=10 divider=512 samples=1024 RO=OFF
CAPTURE 11/16 round=11 divider=512 samples=1024 RO=OFF
CAPTURE 12/16 round=12 divider=512 samples=1024 RO=OFF
CAPTURE 13/16 round=13 divider=512 samples=1024 RO=OFF
CAPTURE 14/16 round=14 divider=512 samples=1024 RO=OFF
CAPTURE 15/16 round=15 di

CAPTURE 16/16 round=16 divider=512 samples=1024 RO=OFF
CAMPAIGN 768/1000 batches saved; RO OFF=True
Saving to health_assessment/2026-11-11_20261110T192028Z_7e1391ad/segments/board_21600/2026-11-11_20261110T192143Z_b161a4a3
CAPTURE 1/16 round=1 divider=512 samples=1024 RO=OFF
CAPTURE 2/16 round=2 divider=512 samples=1024 RO=OFF
CAPTURE 3/16 round=3 divider=512 samples=1024 RO=OFF
CAPTURE 4/16 round=4 divider=512 samples=1024 RO=OFF
CAPTURE 5/16 round=5 divider=512 samples=1024 RO=OFF
CAPTURE 6/16 round=6 divider=512 samples=1024 RO=OFF
CAPTURE 7/16 round=7 divider=512 samples=1024 RO=OFF
CAPTURE 8/16 round=8 divider=512 samples=1024 RO=OFF
CAPTURE 9/16 round=9 divider=512 samples=1024 RO=OFF
CAPTURE 10/16 round=10 divider=512 samples=1024 RO=OFF
CAPTURE 11/16 round=11 divider=512 samples=1024 RO=OFF
CAPTURE 12/16 round=12 divider=512 samples=1024 RO=OFF
CAPTURE 13/16 round=13 divider=512 samples=1024 RO=OFF
CAPTURE 14/16 round=14 divider=512 samples=1024 RO=OFF
CAPTURE 15/16 round=15 di

CAPTURE 16/16 round=16 divider=512 samples=1024 RO=OFF
CAMPAIGN 896/1000 batches saved; RO OFF=True
Saving to health_assessment/2026-11-11_20261110T192028Z_7e1391ad/segments/board_21600/2026-11-11_20261110T192155Z_3f9c7e3e
CAPTURE 1/16 round=1 divider=512 samples=1024 RO=OFF
CAPTURE 2/16 round=2 divider=512 samples=1024 RO=OFF
CAPTURE 3/16 round=3 divider=512 samples=1024 RO=OFF
CAPTURE 4/16 round=4 divider=512 samples=1024 RO=OFF
CAPTURE 5/16 round=5 divider=512 samples=1024 RO=OFF
CAPTURE 6/16 round=6 divider=512 samples=1024 RO=OFF
CAPTURE 7/16 round=7 divider=512 samples=1024 RO=OFF
CAPTURE 8/16 round=8 divider=512 samples=1024 RO=OFF
CAPTURE 9/16 round=9 divider=512 samples=1024 RO=OFF
CAPTURE 10/16 round=10 divider=512 samples=1024 RO=OFF
CAPTURE 11/16 round=11 divider=512 samples=1024 RO=OFF
CAPTURE 12/16 round=12 divider=512 samples=1024 RO=OFF
CAPTURE 13/16 round=13 divider=512 samples=1024 RO=OFF
CAPTURE 14/16 round=14 divider=512 samples=1024 RO=OFF
CAPTURE 15/16 round=15 di

## 5. 離線匯出，不操作硬體
將 payload 匯出聯合資料、逐通道資料，以及每次啟動檢查完成後的前 1000 個可用樣本。不是整板上電矩陣。這格不做熵認證。

In [6]:
dataset = export_datasets(campaign["folder"])
print(json.dumps(dataset, indent=2))
assert dataset["total_samples"] == 1024000

{
  "schema": 1,
  "integrity": "PASS",
  "board_id": "21600",
  "build_id": "entropy-health-20261110T172050Z",
  "divider": 512,
  "batches": 1000,
  "total_samples": 1024000,
  "restart_rows": 1000,
  "restart_columns": 1000,
  "joint_format": "one byte per time point; bits 0..7 = simultaneous RO channels",
  "channel_format": "one byte per sample, value 0 or 1; NOT packed bits",
  "restart_format": "row-major; first 1000 PAYLOAD samples after 1024 startup health samples per RO restart",
  "continuous": false,
  "entropy_estimate": null,
  "physical_entropy_certified": false,
  "limitations": "No entropy estimate/certification. RO batch restart is not board power cycling. Do not sum channel entropy. Preserve all batch boundaries and source records."
}


## 6. 打包下載
下載 ZIP 與有執行輸出的本 notebook。只完成第一段也可執行；若第二段中斷未回傳 campaign，請將畫面印出的 Campaign folder 路徑以 Path 加入 folders，或先提供錯誤畫面協助保存。不要重新跑硬體格。

In [7]:
folders = [functional_folder]
if "campaign" in globals():
    folders.append(Path(campaign["folder"]))
zip_path = Path("entropy_health_" + functional_folder.name + ".zip")
with zipfile.ZipFile(str(zip_path), "w", zipfile.ZIP_DEFLATED) as z:
    for folder in folders:
        for p in sorted(folder.rglob("*")):
            if p.is_file():
                z.write(str(p), str(Path(folder.parent.name) / folder.name / p.relative_to(folder)))
print("Download:", str(zip_path))
print("No more hardware actions needed")

Download: entropy_health_2026-11-11_20261110T192024Z_c6d5135c.zip
No more hardware actions needed
